#BERTopic on UNTM Datasets
In this notebook we have performed hyper-parameter tuning on Urdu News Data using a bert based topic modelling technique BERTopic.

## Mounting Google Drive
If the dataset is on Google Drive then you have to mount over google drive with collaboratory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive




#Installing required dependencies
**One thing to remember is that after installing libraries you have to restart the run time again so that other dependencies are not affected by it.**

In [ ]:
!pip install bertopic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.1/154.1 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 9.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.9/90.9 kB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 9.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached Cython-0.29.36-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.manylinux_2_24_x86_64.whl (1.9 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 6.2 MB/s eta 0:00:00
  Created wheel for hdbscan: filename=hdbscan-0.8.33-cp310-cp310-linux_x86_64.whl size=3039181 sha256=f37fe05d923082d79c2f749ade73e29e1b536066f51b996d6f3db56a73e9cd19
  Stored in direc

In [ ]:
!pip install -U sentence-transformers

In [ ]:
!pip install --upgrade tensorflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.2/475.2 MB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 75.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 37.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 71.3 MB/s eta 0:00:00
  Attempting uninstall: tensorflow-estimator
    Found existing installation: tensorflow-estimator 2.12.0
    Uninstalling tensorflow-estimator-2.12.0:
      Successfully uninstalled tensorflow-estimator-2.12.0
  Attempting uninstall: keras
    Found existing installation: keras 2.12.0
    Uninstalling keras-2.12.0:
      Successfully uninstalled keras-2.12.0
  Attempting uninstall: google-auth-oauthlib
    Found existing installation: google-auth-oauthlib 0.4.6
    Uninstalling google-auth-oauthlib-0.4.6:
      Successfully uninstalled google-auth-oauthlib-0.4.6
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.12.0
    Uninstalling te


# Importing required dependencies
We will import numpy, pandas and re, bertopic, gensim library for now. other libraries will be imported in the notebook later.

Pandas will be used to create a Dataframe and handle the csv file. Numpy will be used for the faster computation of arrays to save time. re library will be used for the cleaning of data. gensim library is used to get coherence score. Bertopic is used to train bertopic on our UNTM dataset with using pretrained language models

In [ ]:
import pandas as pd
import numpy as np
import re
from bertopic import BERTopic
from gensim.models.coherencemodel import CoherenceModel
import gensim.corpora as corpora
#optional
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.ERROR)

import warnings
warnings.filterwarnings("ignore",category=DeprecationWarning)

##DataFrame
We scraped 8001 urdu news that consist of 7 categories=>Business,Technology, showbiz,Health,Politics,weird and Sports. After cleaning news, total 7991 rows saved into another new column Clean_Data for experiments.



In [ ]:
#load UNTM Dataset
df = pd.read_csv('/content/drive/MyDrive/Clean_UNTM.csv', encoding='utf-8')

In [ ]:
print(len(df))
df.head()

7991


,Date,Title,News,Category,Link,Clean_Data
0,2023-04-25,انوشکا شرما اور ویرات کوہلی کی بیڈ منٹن کھیلتے...,ممبئی(اُردو پوائنٹ اخبارتازہ ترین - آن لائن۔ 2...,Sports,https://www.urdupoint.com/sports/detail-news/l...,بھارتی کرکٹ کے آئیکون ویرات کوہلی اور انکی ...
1,2023-04-25,شکست کا ملبہ کسی پر بھی نہیں ڈالوں گا،بابراعظم,راولپنڈی(اُردو پوائنٹ اخبارتازہ ترین - آن لائن...,Sports,https://www.urdupoint.com/sports/detail-news/l...,پاکستان کرکٹ ٹیم کے کپتان بابر اعظم کا کہنا ...
2,2023-04-25,آئی پی ایل:سلواوورریٹ پر کوہلی کو بھاری جرمان...,نئی دہلی (اُردو پوائنٹ اخبارتازہ ترین - آن لائ...,Sports,https://www.urdupoint.com/sports/detail-news/l...,رائل چیلنجرز بنگلور کے کپتان ویرات کوہلی کو ا...
3,2023-04-25,ٹی ٹونٹی ایک مشکل فارمیٹ ہے ،جیت کیلئے اے گیم ...,راولپنڈی (اُردو پوائنٹ اخبارتازہ ترین - آن لائ...,Sports,https://www.urdupoint.com/sports/detail-news/l...,نیوزی لینڈ سے سیریز نہ جیت پانے پر قومی ٹیم ...
4,2023-04-25,آئرلینڈ نے ٹیسٹ کرکٹ میں اپنا سب سے زیادہ ٹیس...,گال(اُردو پوائنٹ اخبارتازہ ترین - آن لائن۔ 25 ...,Sports,https://www.urdupoint.com/sports/detail-news/l...,سری لنکا اور آئرلینڈ کے درمیان جاری دو ٹیسٹ...


## Preprocessing of Data
Data was cleaned that saved into column "Clean_Data" we  removed some extra coloumns first like date,Title,Link and some other columns too. Because we do experiment only news  and all other things are extra for us.

Stopwords are common words that are often filtered out during text processing in natural language processing (NLP) tasks. These words are considered to have little or no value in conveying the actual meaning of the text. We take list of 401 stopwords for topic modelling.

In [ ]:
df=df.drop(columns=['Date','Title', 'Link'])

KeyError: ignored

In [ ]:
def remove_nonbreaking_space(text):
    return re.sub(r'\xa0', ' ', text)

df['Clean_Data'] = df['Clean_Data'].apply(remove_nonbreaking_space)

In [ ]:
df

,News,Category,Clean_Data
0,ممبئی(اُردو پوائنٹ اخبارتازہ ترین - آن لائن۔ 2...,Sports,بھارتی کرکٹ کے آئیکون ویرات کوہلی اور انکی ...
1,راولپنڈی(اُردو پوائنٹ اخبارتازہ ترین - آن لائن...,Sports,پاکستان کرکٹ ٹیم کے کپتان بابر اعظم کا کہنا ...
2,نئی دہلی (اُردو پوائنٹ اخبارتازہ ترین - آن لائ...,Sports,رائل چیلنجرز بنگلور کے کپتان ویرات کوہلی کو ا...
3,راولپنڈی (اُردو پوائنٹ اخبارتازہ ترین - آن لائ...,Sports,نیوزی لینڈ سے سیریز نہ جیت پانے پر قومی ٹیم ...
4,گال(اُردو پوائنٹ اخبارتازہ ترین - آن لائن۔ 25 ...,Sports,سری لنکا اور آئرلینڈ کے درمیان جاری دو ٹیسٹ...
...,...,...,...
7986,پشاور(اُردو پوائنٹ اخبارتازہ ترین - اے پی پی۔ ...,Health,پشاور کے لیڈی ریڈنگ ہسپتال میں کورونا وائرس ...
7987,بیروت (اُردو پوائنٹ اخبارتازہ ترین - این این آ...,Health,لبنان کی وزارت صحت نے کووِڈکی اومیکرون قسم کے...
7988,نیویارک(اُردو پوائنٹ اخبارتازہ ترین - این این ...,Health,عالمی ادارہ صحت کے ویکسین ایڈوائزری پینل نے ص...
7989,بھکر۔(اُردو پوائنٹ اخبارتازہ ترین - اے پی پی۔ ...,Health,ڈپٹی کمشنر حامد محمود ملہی کے زیر صدارت ضلعی...


In [ ]:
documents = df['Clean_Data'].values.tolist()
print((documents[1]))

  پاکستان کرکٹ ٹیم کے کپتان بابر اعظم کا کہنا ہے کہ شکست کا ملبہ کسی پر بھی نہیں ڈالوں گا کیونکہ یہ ٹیم ورک ہوتا ہے اور ہمیشہ ایک ٹیم جیتتی ہے یا ہارتی ہے تاہم جہاں ضرورت ہے اچھا کرنے کی مزید بہتری ضرور لائیں گے۔راولپنڈی میں ٹی  میچز کی سیریز مکمل ہونے کے بعد پاکستان ٹیم کے کپتان بابر اعظم کا کہنا تھا کہ شروع میں اندازہ تھا کہ  سکور کریں گے لیکن  رنز کا اچھا سکور کیا لیکن نیوزی لینڈ کے کھلاڑی اچھا کھیلے۔ میں کبھی مطمئن نہیں ہوتا اور تربیت بھی یہی ہے کہ اچھا کرکے بھی سیکھا جاتا ہے کیونکہ بری پرفارمنس کے بعد تو ہر کوئی سیکھتا ہے۔بابر اعظم کا کہنا تھا کہ دو دن کے وقفے سے کوئی فرق نہیں پڑتا  اوور کے بعد  کا ہدف سمجھ رہے تھے لیکن یہ درست ہے کہ آخری تین اوور میں رضوان سے ہٹ نہیں لگی رضوان مزید دو رنز کر سکتے تھے لیکن نہیں کرسکے بہرحال رضوان نے بہت اچھا کھیلا کیچز ڈراپ ہوئے جو کہ نہیں ہونے چاہیے تھے۔


In [ ]:
import nltk
stopwords=pd.read_csv('/content/drive/MyDrive/stopwords.txt',names=['List'])
# stopwords['List']

stopwords_ur=[]
for li in stopwords['List']:
  stopwords_ur.append(li)
print(len(stopwords_ur))
print(stopwords_ur)

401
['کی', 'ہیں', 'ہے', 'رہا', 'رہی', 'رہے', 'تھا', 'تھے', 'تھی', 'تھى', 'میں', 'کہ', 'کے', 'ہوتے', 'کہہ', 'بنانا', 'پھر', 'لکین', 'ہوتی', 'لیتی', 'ایسی', 'ائے', 'ہوئے', 'ہوۓ ', 'مگر', 'چاہے', 'کیے', 'تاکہ', 'تم', 'آکر', 'لگا', 'ہوگیئں', 'ليے', 'رہتی', 'ہوگئی', 'انھوں', 'چاہتے', 'پاگیئں', 'آنا', 'کروا', 'رکھ', 'آتی', 'یہاں', 'جیسا', 'چکے', 'کرئے', 'دیے', 'چکا', 'ملتا', 'کبھی', 'ایسا', 'کرسکیں', 'ہوسکے', 'سکیں', 'لہذا', 'چاہئے', 'وہیں', 'اٹھایا', 'جہنوں', 'ہمارے', 'لیے', 'آرہے', 'کرگیئں', 'بنانے', 'سکتا', 'وغیرہ', 'دے', 'ہوۓ', 'رہنے', 'ہوۓ', 'کئے', 'لگے', 'لگایا', 'لائے', 'کہے', 'کرے', 'رہئں', 'آگئے', 'کئی', 'کم', 'ملی', 'جنہیں', 'ہوئیں', 'تھیں', 'ابھی', 'پاگئیں', 'آئے', 'کرا', 'دیا', 'جہاں', 'آگئیں', 'کرتی', 'رہیں', 'کرتیں', 'دیتی', 'ہوگئے', 'ديتے', 'انہں', 'ایسے', 'رکھتے', 'رہتے', 'رکھی', 'کیں', 'كیا', 'وه', 'جنہيں', 'کروائی', 'دینا', 'جسے', 'جاتی', 'اسکی', 'ملے', 'کرگئى', 'جن', 'آپکو', 'جنہوں', 'دیئے', 'والی', 'جبکہ', 'دیگر', 'آپکا', 'رکھنے', 'انہىں', 'کيے', 'یعنی', 'كيے', 'بننے', 'ج

# BERTopic Training
The default  bertopic embedding model is paraphrase-multilingual-MiniLM-L12-v2 when selecting language="multilingual". we also performed experiment on 2 another pretained multilingual model from [sentence-tranformer](https://www.sbert.net/docs/pretrained_models.html) and create custom document embedding and passed it to the bertopic model for training.

In [ ]:
#create custom embedding
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
embeddings = model.encode(documents, show_progress_bar=True)
print(embeddings)

.gitattributes:   0%|          | 0.00/968 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.09k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

unigram.json:   0%|          | 0.00/14.8M [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

Batches:   0%|          | 0/250 [00:00<?, ?it/s]

[[ 0.07987808 -0.07742516 -0.08378563 ... -0.043457   -0.11484845
  -0.07181289]
 [ 0.09646066 -0.04020533 -0.04377843 ...  0.00730114  0.09356619
   0.05510345]
 [ 0.00332669  0.09647436 -0.17899874 ... -0.15398908  0.01223139
  -0.0459842 ]
 ...
 [-0.05227447  0.06783362 -0.2360085  ... -0.18109454 -0.0346763
   0.08866747]
 [ 0.1034833   0.31677362  0.00757494 ...  0.09555458  0.02051694
   0.20520177]
 [-0.09281214  0.19791466 -0.18801452 ... -0.22458135 -0.11964664
   0.11853428]]


In [ ]:
#pass vectorizer_model to bertopic for stopwords removal
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(stop_words= stopwords_ur)


In [ ]:
#ClassTfidf for top words
from bertopic.vectorizers import ClassTfidfTransformer

ctfidf_model = ClassTfidfTransformer(bm25_weighting=True, reduce_frequent_words=True)

In [ ]:
#UMAP for dimention reduction
from umap import UMAP

In [ ]:
from hdbscan import HDBSCAN

In [ ]:
np.random.seed(42)

**Diversity Score**

In [ ]:
def topic_diversity(topics, topk=10):
    """
    compute the proportion of unique words

    Parameters
    ----------
    topics: a list of lists of words
    topk: top k words on which the topic diversity will be computed
    """
    if topk > len(topics[0]):
        raise Exception('Words in topics are less than '+str(topk))
    else:
        unique_words = set()
        for topic in topics:
            unique_words = unique_words.union(set(topic[:topk]))
        puw = len(unique_words) / (topk * len(topics))
        return puw

In [ ]:
import itertools
from rbo import rbo
import numpy as np

class InvertedRBO:
    def __init__(self):
        pass

    def irbo(self, topics, topk=10, weight=0.9):
        """
        Calculate inverted Rank Biased Overlap (RBO) as a measure of topic diversity from a list of lists of words.

        :param topics: A list of lists of words representing different topics.
        :param topk: The number of top words on which RBO will be computed.
        :param weight: Weight of each agreement at depth d: p**(d-1). When set to 1.0, there is no weight,
                       and the RBO returns to average overlap.
        :return: The inverted RBO topic diversity score.
        """
        if topk <= 0:
            raise ValueError("topk must be a positive integer.")

        num_topics = len(topics)
        if num_topics == 0:
            raise ValueError("topics list cannot be empty.")

        if topk > len(topics[0]):
            raise Exception('Words in topics are less than topk')

        collect = []
        for list1, list2 in itertools.combinations(topics, 2):
            rbo_val = rbo(list1[:topk], list2[:topk], p=weight)[2]
            collect.append(rbo_val)

        Irbo_score = 1 - np.mean(collect)
        return Irbo_score

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from itertools import product
from sklearn.cluster import AgglomerativeClustering

# cluster_model = AgglomerativeClustering(n_clusters=7)
# cluster_model = KMeans(n_clusters=7, random_state=42)
texts = [[word for word in str(document).split() if word not in stopwords_ur] for document in documents]
id2word = corpora.Dictionary(texts)
corpus = [id2word.doc2bow(text) for text in texts]

# Define lists for n_neighbors and n_components
n_neighbors_list = [5, 10,15,20,25,30]
n_components_list = [2, 3, 4, 5]

# Create a list to store the data for each combination
data_list = []

# Loop over the parameter combinations and fit BERTopic models
for n_neighbors, n_components in product(n_neighbors_list, n_components_list):
    umap_param = {'n_neighbors': n_neighbors, 'n_components': n_components}

    # Create UMAP and HDBSCAN models with the current parameter combination
    umap_model = UMAP(**umap_param, random_state=42)

    # Fit a BERTopic model with the current parameter combination
    topic_model = BERTopic(language="urdu",
                          low_memory=True,
                          calculate_probabilities=True,
                          vectorizer_model=vectorizer_model,
                          top_n_words=10,
                          ctfidf_model=ctfidf_model,
                          verbose=True, umap_model=umap_model, nr_topics=8)

    topics, probs = topic_model.fit_transform(documents, embeddings)

    topics_bert = []
    for i in topic_model.get_topics():
        row = []
        topic = topic_model.get_topic(i)
        for word in topic:
            row.append(word[0])
        topics_bert.append(row)

    # Calculate coherence scores using Gensim's Coherence Model
    coherence_cv = CoherenceModel(topics=topics_bert, texts=texts, dictionary=id2word, coherence='c_v')
    cv_score = round(coherence_cv.get_coherence(),3)
    coherence_npmi = CoherenceModel(topics=topics_bert, texts=texts, dictionary=id2word, coherence='c_npmi')
    npmi_score = round(coherence_npmi.get_coherence(),3)

    # Calculate Unique Word count
    unique_words = round(topic_diversity(topics_bert, topk=10),3)

    # Calculate IRBO Score
    inverted_rbo_calculator = InvertedRBO()
    IRBO = round(inverted_rbo_calculator.irbo(topics_bert, topk=10, weight=0.9),3)

    # Append the data for the current combination to the list
    data_list.append({
        'Clustering': 'HDBSCAN',
        'Pretrained_Model': 'MiniLM',
        'n_neighbors': n_neighbors,
        'n_components': n_components,
        'C_v score': cv_score,
        'C_NPMI score': npmi_score,
        'Unique Word': unique_words,
        'IRBO Score': IRBO,
        'Topics': topics_bert
    })

# Create a DataFrame from the data list
df1 = pd.DataFrame(data_list)

# Print the DataFrame
print(df1)


2023-11-29 17:02:31,267 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2023-11-29 17:02:42,293 - BERTopic - Dimensionality - Completed ✓
2023-11-29 17:02:42,299 - BERTopic - Cluster - Start clustering the reduced embeddings
2023-11-29 17:02:52,001 - BERTopic - Cluster - Completed ✓
2023-11-29 17:02:52,003 - BERTopic - Representation - Extracting topics from clusters using representation models.
2023-11-29 17:02:56,471 - BERTopic - Representation - Completed ✓
2023-11-29 17:02:56,476 - BERTopic - Topic reduction - Reducing number of topics
2023-11-29 17:03:02,610 - BERTopic - Topic reduction - Reduced number of topics from 137 to 8
2023-11-29 17:03:23,929 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2023-11-29 17:03:36,544 - BERTopic - Dimensionality - Completed ✓
2023-11-29 17:03:36,547 - BERTopic - Cluster - Start clustering the reduced embeddings
2023-11-29 17:03:43,778 - BERTopic - Cluster - Completed ✓
2023-11-29 17:03:4

   Clustering Pretrained_Model  n_neighbors  n_components  C_v score  \
0     HDBSCAN           MiniLM            5             2      0.625   
1     HDBSCAN           MiniLM            5             3      0.601   
2     HDBSCAN           MiniLM            5             4      0.672   
3     HDBSCAN           MiniLM            5             5      0.670   
4     HDBSCAN           MiniLM           10             2      0.641   
5     HDBSCAN           MiniLM           10             3      0.645   
6     HDBSCAN           MiniLM           10             4      0.682   
7     HDBSCAN           MiniLM           10             5      0.607   
8     HDBSCAN           MiniLM           15             2      0.632   
9     HDBSCAN           MiniLM           15             3      0.684   
10    HDBSCAN           MiniLM           15             4      0.598   
11    HDBSCAN           MiniLM           15             5      0.667   
12    HDBSCAN           MiniLM           20             2      0

In [ ]:
# Save the DataFrame to a CSV file
df1.to_csv('UNTM_HDBSCAN_ClusteringF.csv', mode='a', index=False,  encoding='utf-8-sig')